
## E-Commerce Order, Refund and Exception Handling System

**Name:** Venkat Phani Kiran  
**Use case:** E-Commerce — Order, Refund & Exception Handling  
**Framework:** CrewAI (four collaborating agents, sequential process)  
**Models:** Together AI, configured through `.env`

In [1]:
# Install project dependencies
%pip install -q -U crewai python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Configure Together AI models for CrewAI
# Model ids, API key and base URL are read from the .env file in the project folder.

import os
import logging
from dotenv import load_dotenv, find_dotenv
from crewai import LLM

ENV_PATH = find_dotenv(usecwd=True) or "/content/.env"
load_dotenv(ENV_PATH, override=True)

API_KEY         = os.getenv("TOGETHER_API_KEY")
BASE_URL        = os.getenv("TOGETHER_BASE_URL", "https://api.together.xyz/v1")
PRIMARY_MODEL   = os.getenv("TOGETHER_MODEL")
FALLBACK_MODEL  = os.getenv("LLM_MODEL_FALLBACK")
REASONING_MODEL = os.getenv("LLM_MODEL_REASONING", PRIMARY_MODEL)

if not API_KEY or not PRIMARY_MODEL:
    raise ValueError(f"TOGETHER_API_KEY and TOGETHER_MODEL must be set in {ENV_PATH}")

os.environ["OPENAI_API_KEY"]  = API_KEY
os.environ["OPENAI_API_BASE"] = BASE_URL
os.environ["CREWAI_TELEMETRY_OPT_OUT"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
logging.getLogger("crewai").setLevel(logging.ERROR)

def create_llm(model_name: str, max_tokens: int = 4000) -> LLM:
    """Build a CrewAI LLM that talks to Together's OpenAI-compatible chat endpoint."""
    return LLM(
        model=f"openai/{model_name}",
        base_url=BASE_URL,
        api_key=API_KEY,
        temperature=0,
        max_tokens=max_tokens,
        timeout=90,
        custom_openai=True,
    )

primary_llm   = create_llm(PRIMARY_MODEL)
fallback_llm  = create_llm(FALLBACK_MODEL) if FALLBACK_MODEL else None
reasoning_llm = create_llm(REASONING_MODEL, max_tokens=6000)

print("Environment file :", ENV_PATH)
print("Primary model    :", PRIMARY_MODEL)
print("Fallback model   :", FALLBACK_MODEL)
print("Reasoning model  :", REASONING_MODEL)

Environment file : /Users/emk180/Downloads/.env
Primary model    : deepseek-ai/DeepSeek-V4.1-Flash
Fallback model   : zai-org/GLM-5.3-Flash
Reasoning model  : deepseek-ai/DeepSeek-V4.1-Flash


In [3]:
# Connectivity check for the configured models

def smoke_test(name: str, model) -> bool:
    if model is None:
        print(f"{name:<9}: not configured")
        return False
    try:
        reply = model.call("Reply with exactly: READY")
        print(f"{name:<9}: OK  -> {str(reply).strip()[:60]!r}")
        return True
    except Exception as error:
        print(f"{name:<9}: FAILED -> {str(error)[:300]}")
        return False

ok_primary   = smoke_test("primary",   primary_llm)
ok_fallback  = smoke_test("fallback",  fallback_llm)
ok_reasoning = smoke_test("reasoning", reasoning_llm)

if not (ok_primary or ok_fallback):
    raise RuntimeError("No model responded. Check TOGETHER_API_KEY and the model ids in .env.")

primary  : OK  -> 'READY'
fallback : OK  -> 'READY'
reasoning: OK  -> 'READY'


## Section A — Problem Definition

**Industry:** E-Commerce

**Project title:** Order, Refund and Exception Handling Assistant

**Problem:** An online store receives a steady stream of post-purchase queries: delayed deliveries, refund requests, damaged or wrong items and return-eligibility questions. Support agents must look up the order, apply the return and refund policy correctly, decide on a resolution, and recognise the cases that need a human approver — high-value refunds, policy exceptions, disputed deliveries and possible refund abuse.

**Objective:** Build a goal-driven multi-agent system that takes a customer query plus an order id, identifies the issue, interprets the applicable policy, recommends a resolution with the exact refund or replacement action, and decides whether the resolution can be applied automatically or needs manual approval.

**Architecture:** Four CrewAI agents in a sequential crew. Each task receives the earlier tasks' outputs as context, and every agent returns a JSON record with fixed fields so that downstream agents work from structured data. Together AI models do the reading, policy reasoning and writing; deterministic Python tools supply order data, policy text and the approval rules.

**Design principle:** the LLM reasons, the rule engine decides. Approval thresholds live in code, so identical cases always receive the same decision, and every decision carries the reasons that produced it.

**Scope:** Simulated orders and customers of a fictional store ("KartNow"), a static policy, and a simulated approval queue. No live systems are touched. Knowledge sources are static, so no retrieval-augmented generation is used.

## Section B — Data, Tools and Agents

### B.1 Simulated knowledge sources
An order database (what the order-management system would return) and the store's return and refund policy, both hard-coded.

In [4]:
# Mock data: orders and the return / refund policy

import json, re
from datetime import datetime, timezone, date

STORE = "KartNow"
TODAY = date(2026, 11, 9)

# --- 1. Orders (what the order-management system would return) -------------
ORDERS = {
    "ORD-1001": {"customer": "Riya Sen",      "item": "Ceramic dinner set (16 pc)", "item_category": "home",
                 "amount_inr": 2199, "order_date": "2026-11-01", "estimated_delivery": "2026-11-05",
                 "delivered_date": None, "status": "in_transit", "payment_method": "UPI", "refunds_last_90d": 0},
    "ORD-1002": {"customer": "Arjun Mehta",   "item": "Cotton T-shirt, size M",      "item_category": "apparel",
                 "amount_inr": 1299, "order_date": "2026-10-28", "estimated_delivery": "2026-11-01",
                 "delivered_date": "2026-11-01", "status": "delivered", "payment_method": "card", "refunds_last_90d": 0},
    "ORD-1003": {"customer": "Fatima Khan",   "item": "Kitchen blender 750W",        "item_category": "appliances",
                 "amount_inr": 3499, "order_date": "2026-11-02", "estimated_delivery": "2026-11-05",
                 "delivered_date": "2026-11-05", "status": "delivered", "payment_method": "UPI", "refunds_last_90d": 1},
    "ORD-1004": {"customer": "Dev Nair",      "item": "Wireless headphones",         "item_category": "electronics",
                 "amount_inr": 2499, "order_date": "2026-10-15", "estimated_delivery": "2026-10-18",
                 "delivered_date": "2026-10-18", "status": "delivered", "payment_method": "card", "refunds_last_90d": 0},
    "ORD-1005": {"customer": "Priya Iyer",    "item": "Laptop 14-inch, 16 GB RAM",   "item_category": "electronics",
                 "amount_inr": 68999, "order_date": "2026-11-04", "estimated_delivery": "2026-11-07",
                 "delivered_date": "2026-11-07", "status": "delivered", "payment_method": "card", "refunds_last_90d": 0},
    "ORD-1006": {"customer": "Sameer Joshi",  "item": "Smartwatch",                  "item_category": "electronics",
                 "amount_inr": 4999, "order_date": "2026-11-03", "estimated_delivery": "2026-11-06",
                 "delivered_date": "2026-11-06", "status": "delivered", "payment_method": "COD", "refunds_last_90d": 3},
    "ORD-1007": {"customer": "Meera Pillai",  "item": "Table lamp",                  "item_category": "home",
                 "amount_inr": 1599, "order_date": "2026-11-03", "estimated_delivery": "2026-11-06",
                 "delivered_date": "2026-11-06", "status": "delivered", "payment_method": "UPI", "refunds_last_90d": 0},
}

# --- 2. Return and refund policy (static) ----------------------------------
RETURN_WINDOWS_DAYS = {"apparel": 30, "home": 30, "appliances": 15, "electronics": 15, "perishable": 0}

POLICY_TEXT = """
KartNow Return and Refund Policy (excerpt)
1. Return windows start on the delivery date: apparel 30 days, home 30 days, appliances 15 days,
   electronics 15 days. Perishables cannot be returned.
2. Damaged, defective or wrong items reported within the return window qualify for a free
   replacement or a full refund, at the customer's choice. Photo or video evidence is requested
   for damaged and wrong-item claims.
3. Refunds go back to the original payment method within 5-7 working days; cash-on-delivery
   orders are refunded to the customer's KartNow wallet or bank account.
4. A delivery is treated as delayed when it is more than 2 days past the estimated date. Delayed
   orders are not refunded while in transit; the customer may cancel for a full refund if the
   order has not shipped, or refuse delivery.
5. Requests outside the return window are policy exceptions and need manual approval.
6. Refunds above Rs 5,000 need approval by a senior agent; refunds above Rs 15,000 need approval
   by a team lead.
7. "Item not received" claims where the courier shows delivered are disputes and need manual review.
8. Customers with 3 or more refunds in the last 90 days are referred to the fraud-review team
   before any further refund is issued.
"""

print(f"Loaded {len(ORDERS)} orders and the {STORE} return policy.")

Loaded 7 orders and the KartNow return policy.


### B.2 Approval policy as code
Whether a resolution can be applied automatically is a deliberate decision. The rule engine takes the structured outputs of the Issue Identification, Policy Interpretation and Resolution agents together with the order record, and returns one of three decisions with a risk score and reasons.

**Hard override:** customer has 3 or more refunds in the last 90 days and the query concerns a refund, return, damaged or wrong item, or non-receipt → `ESCALATE_FRAUD_REVIEW`.

**Risk score** (additive) otherwise:

| Signal | Points | Source |
|---|---|---|
| Refund or replacement value above Rs 15,000 / above Rs 5,000 | +60 / +40 | Resolution + order |
| Request is outside the return window (policy exception) | +30 | Policy |
| Identification confidence below 0.6 (vague or incomplete query) | +30 | Identification |
| "Not received" claim while courier shows delivered (dispute) | +30 | Identification + order |
| Damaged / wrong-item claim with no photo or video evidence | +20 | Identification |

**Thresholds:** score ≥ 30 → `MANUAL_APPROVAL` (approver: team lead above Rs 15,000, otherwise senior agent); below 30 → `AUTO_APPROVE`.

In [5]:
# Deterministic approval rule engine

MONEY_ACTIONS = {"full_refund", "partial_refund", "refund_and_return", "replacement"}
REFUND_CATEGORIES = {"refund_request", "return_request", "damaged_product", "wrong_item", "not_received"}

def days_between(a: str, b: date) -> int:
    return (b - date.fromisoformat(a)).days

def evaluate_approval_rules(identification: dict, policy: dict, resolution: dict, order: dict) -> dict:
    """Apply the KartNow approval policy. Pure function: same inputs -> same decision."""
    reasons = []
    action = str(resolution.get("recommended_action", "")).lower().replace(" ", "_")
    try:
        value = float(resolution.get("refund_amount_inr") or 0)
    except (TypeError, ValueError):
        value = 0.0
    try:
        confidence = float(identification.get("confidence", 0.5))
    except (TypeError, ValueError):
        confidence = 0.5
    category = str(identification.get("issue_category", "")).lower()
    evidence = identification.get("evidence_provided") is True
    within_window = policy.get("within_return_window")
    exception = policy.get("policy_exception_requested") is True or within_window is False
    refunds_90d = int(order.get("refunds_last_90d", 0) or 0)
    courier_delivered = order.get("status") == "delivered"

    # --- Hard override ----------------------------------------------------
    money_related = action in MONEY_ACTIONS or value > 0 or category in REFUND_CATEGORIES
    if refunds_90d >= 3 and money_related:
        return {"decision": "ESCALATE_FRAUD_REVIEW", "approver": "fraud-review team", "risk_score": 100,
                "reasons": [f"Policy rule 8: {refunds_90d} refunds in the last 90 days"], "override": "fraud"}

    # --- Additive risk score ----------------------------------------------
    score = 0
    def add(points, why):
        nonlocal score
        score += points
        reasons.append(f"+{points} {why}")

    if value > 15000:
        add(60, f"value Rs {value:,.0f} above Rs 15,000 (policy rule 6, team lead)")
    elif value > 5000:
        add(40, f"value Rs {value:,.0f} above Rs 5,000 (policy rule 6, senior agent)")
    if exception and action in MONEY_ACTIONS:
        add(30, "request outside the return window (policy rule 5)")
    if confidence < 0.6:
        add(30, f"identification confidence {confidence:.2f} below 0.60")
    if category == "not_received" and courier_delivered:
        add(30, "item-not-received claim while courier shows delivered (policy rule 7)")
    if category in ("damaged_product", "wrong_item") and not evidence:
        add(20, "damaged / wrong-item claim without photo or video evidence (policy rule 2)")

    if score >= 30:
        approver = "team lead" if value > 15000 else "senior agent"
        return {"decision": "MANUAL_APPROVAL", "approver": approver, "risk_score": score, "reasons": reasons, "override": None}
    return {"decision": "AUTO_APPROVE", "approver": None, "risk_score": score,
            "reasons": reasons or ["within policy, value under limit, no risk signals"], "override": None}

# Self-check on two hand-built inputs
_routine = evaluate_approval_rules({"issue_category": "return_request", "confidence": 0.9, "evidence_provided": False},
                                   {"within_return_window": True}, {"recommended_action": "refund_and_return", "refund_amount_inr": 1299},
                                   ORDERS["ORD-1002"])
_fraud = evaluate_approval_rules({"issue_category": "not_received", "confidence": 0.8},
                                 {"within_return_window": True}, {"recommended_action": "full_refund", "refund_amount_inr": 4999},
                                 ORDERS["ORD-1006"])
print("Routine return ->", _routine["decision"], "score", _routine["risk_score"])
print("Repeat refunder ->", _fraud["decision"])

Routine return -> AUTO_APPROVE score 0
Repeat refunder -> ESCALATE_FRAUD_REVIEW


### B.3 Tools
Three deterministic tools. Agents decide when to call them and how to use the result; the tools never guess.

| Tool | Used by | Returns |
|---|---|---|
| `lookup_order` | Issue Identification agent | item, amount, dates, delivery status, payment method, refund history, delay in days |
| `check_return_policy` | Policy Interpretation agent | return window for the item category, days since delivery, whether the request is inside the window, the policy text |
| `evaluate_approval` | Escalation agent | the rule engine's decision, approver, score and reasons |

In [6]:
# Tools (CrewAI @tool wrappers around deterministic functions) + JSON helpers

from typing import Union
from crewai.tools import tool

def extract_json(text):
    """Extract the first JSON object from a model response, tolerating fences, prose,
    line breaks inside strings and minor syntax slips."""
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    candidates = [text]
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match and match.group(0) != text:
        candidates.append(match.group(0))
    for candidate in candidates:
        for strict in (True, False):
            try:
                return json.loads(candidate, strict=strict)
            except json.JSONDecodeError:
                pass
    try:
        from json_repair import repair_json
        repaired = repair_json(candidates[-1], return_objects=True)
        if isinstance(repaired, dict) and repaired:
            return repaired
    except Exception:
        pass
    raise ValueError("No valid JSON object found in model output")

def _to_dict(value, label):
    if isinstance(value, dict):
        return value
    try:
        return extract_json(value)
    except Exception:
        return {"_parse_error": f"{label} was not valid JSON"}

@tool("lookup_order")
def lookup_order(order_id: str) -> str:
    """Look up a KartNow order by id (e.g. 'ORD-1003'). Returns JSON with customer, item,
    item_category, amount_inr, order_date, estimated_delivery, delivered_date, status
    (in_transit | delivered), payment_method, refunds_last_90d, and days_late (days past the
    estimated delivery date if not yet delivered, otherwise 0)."""
    order = ORDERS.get(str(order_id).strip().upper())
    if not order:
        return json.dumps({"error": f"No order found for {order_id}"})
    days_late = 0
    if order["delivered_date"] is None:
        days_late = max(0, days_between(order["estimated_delivery"], TODAY))
    return json.dumps({"order_id": order_id, **order, "days_late": days_late, "today": TODAY.isoformat()})

@tool("check_return_policy")
def check_return_policy(item_category: str, delivered_date: str) -> str:
    """Check the KartNow return policy for an item. item_category: apparel | home | appliances |
    electronics | perishable. delivered_date: ISO date (YYYY-MM-DD) or 'none' if not delivered.
    Returns JSON with return_window_days, days_since_delivery, within_return_window, and the
    full policy text."""
    cat = str(item_category).strip().lower()
    window = RETURN_WINDOWS_DAYS.get(cat)
    result = {"item_category": cat, "return_window_days": window, "today": TODAY.isoformat(), "policy_text": POLICY_TEXT.strip()}
    if window is None:
        result["note"] = "Unknown category; treat as 15-day window"
        window = 15
        result["return_window_days"] = window
    if str(delivered_date).lower() in ("none", "null", ""):
        result.update({"days_since_delivery": None, "within_return_window": None,
                       "note": "Not delivered yet; return window has not started"})
    else:
        since = days_between(str(delivered_date), TODAY)
        result.update({"days_since_delivery": since, "within_return_window": (window > 0 and since <= window)})
    return json.dumps(result)

@tool("evaluate_approval")
def evaluate_approval(identification_json: Union[str, dict], policy_json: Union[str, dict],
                      resolution_json: Union[str, dict], order_id: str) -> str:
    """Apply the KartNow approval policy. Pass the Identification, Policy and Resolution JSON
    outputs and the order_id. Returns JSON with decision (AUTO_APPROVE | MANUAL_APPROVAL |
    ESCALATE_FRAUD_REVIEW), approver, risk_score and reasons."""
    identification = _to_dict(identification_json, "identification_json")
    policy = _to_dict(policy_json, "policy_json")
    resolution = _to_dict(resolution_json, "resolution_json")
    order = ORDERS.get(str(order_id).strip().upper(), {})
    return json.dumps(evaluate_approval_rules(identification, policy, resolution, order))

TOOLS = [lookup_order, check_return_policy, evaluate_approval]
print("Tools registered:", [t.name for t in TOOLS])

Tools registered: ['lookup_order', 'check_return_policy', 'evaluate_approval']


### B.4 Agents, tasks and crew
Four agents run as one sequential crew. Each task names the earlier tasks it depends on in `context=[...]`, so every agent works from the structured output of the agents before it.

```
Query + order id ─► 1. Order Issue Identification ─► 2. Policy Interpretation ─► 3. Resolution Recommendation ─► 4. Escalation ─► Audit
                       lookup_order                    check_return_policy            no tools; proposes the           evaluate_approval       rule engine
                                                                                      action, amount and message       (reasoning model)       re-run in Python
```

The Escalation agent decides whether the recommended resolution is applied immediately or routed for manual approval, and adjusts the customer message accordingly, so the customer is never promised a refund that still needs sign-off.

In [7]:
# Agents, tasks and crew builder

from crewai import Agent, Task, Crew, Process

VERBOSE = False   # True prints agent reasoning and tool calls

ISSUE_CATEGORIES = "delayed_delivery | return_request | refund_request | damaged_product | wrong_item | not_received | other"
ACTIONS = "track_and_wait | cancel_and_refund | refund_and_return | full_refund | partial_refund | replacement | request_more_information | no_action"

def build_agents(main_llm, reason_llm, verbose=VERBOSE):
    identification_agent = Agent(
        role="Order Issue Identification Agent",
        goal="Turn a customer query into a structured description of the problem, grounded in the actual order record.",
        backstory=(f"You are a {STORE} support specialist who reads every query carefully, always looks up the order first, "
                   "and never assumes facts the customer did not state. Vague queries get a low confidence score."),
        tools=[lookup_order], llm=main_llm, allow_delegation=False, verbose=verbose, max_iter=5,
    )
    policy_agent = Agent(
        role="Policy Interpretation Agent",
        goal="State exactly which return and refund rules apply to this order and whether the customer's request is within policy.",
        backstory=(f"You know the {STORE} return and refund policy and apply it consistently. You check the return window with the "
                   "tool rather than estimating, cite the policy rule numbers you rely on, and flag exceptions clearly."),
        tools=[check_return_policy], llm=main_llm, allow_delegation=False, verbose=verbose, max_iter=5,
    )
    resolution_agent = Agent(
        role="Resolution Recommendation Agent",
        goal="Recommend one concrete resolution that follows the policy interpretation, with the exact amount, and draft the customer message.",
        backstory=(f"You resolve {STORE} tickets. You choose the single most appropriate action, state the refund or replacement value "
                   "in rupees, and write friendly, precise messages that never promise anything the policy does not allow."),
        tools=[], llm=main_llm, allow_delegation=False, verbose=verbose, max_iter=3,
    )
    escalation_agent = Agent(
        role="Escalation Agent",
        goal="Decide whether the recommended resolution can be applied automatically or needs manual approval, and finalise the customer message.",
        backstory=("You own the approval policy. You call the evaluate_approval tool with the three earlier outputs, adopt its decision "
                   "exactly, explain the reasons, and make sure the customer message matches the decision."),
        tools=[evaluate_approval], llm=reason_llm, allow_delegation=False, verbose=verbose, max_iter=5,
    )
    return identification_agent, policy_agent, resolution_agent, escalation_agent


def build_tasks(agents):
    identification_agent, policy_agent, resolution_agent, escalation_agent = agents

    identify = Task(
        description=(
            "Identify the issue in this customer query.\n\n"
            "Ticket id: {ticket_id}\nOrder id: {order_id}\nCustomer message:\n{message}\n\n"
            "1. Call lookup_order with the order id.\n"
            "2. Classify the issue using only what the customer wrote and what the order record shows.\n"
            "3. Set evidence_provided = true only if the customer says they attached or can share photos or video.\n"
            "4. Set confidence between 0 and 1; a query with no clear problem statement must score below 0.6."
        ),
        expected_output=(
            "A single JSON object (no markdown fences, no commentary; every string value on one line, no line breaks) with exactly these fields:\n"
            f"ticket_id (string), order_id (string), issue_category (one of: {ISSUE_CATEGORIES}), "
            "customer_request (one sentence: what the customer wants), evidence_provided (true/false), "
            "confidence (number 0-1), order (object copied from lookup_order: item, item_category, amount_inr, "
            "delivered_date, status, payment_method, refunds_last_90d, days_late)."
        ),
        agent=identification_agent,
    )

    interpret = Task(
        description=(
            "Interpret the return and refund policy for the identified issue.\n"
            "1. Call check_return_policy with the order's item_category and delivered_date (use 'none' if not delivered).\n"
            "2. Decide whether the customer's request is within policy, and which rule numbers apply.\n"
            "3. Set policy_exception_requested = true if the customer asks for something the policy does not allow "
            "(for example a refund after the return window has closed).\n"
            "4. List the resolution options the policy permits for this case."
        ),
        expected_output=(
            "A single JSON object (no markdown fences; every string value on one line, no line breaks) with exactly these fields:\n"
            "return_window_days (number), days_since_delivery (number or null), within_return_window (true/false/null), "
            "policy_exception_requested (true/false), applicable_rules (list of rule numbers as strings), "
            "permitted_resolutions (list of strings), policy_summary (two sentences in plain language)."
        ),
        agent=policy_agent, context=[identify],
    )

    recommend = Task(
        description=(
            "Recommend the resolution for ticket {ticket_id}.\n"
            "1. Pick exactly one recommended_action that the policy interpretation permits. If the request is outside policy, "
            "still recommend the action the customer would receive if an approver grants the exception, and say so in the rationale.\n"
            "2. Set refund_amount_inr to the order amount for full refunds, returns and replacements (replacement value counts), "
            "a stated partial figure for partial refunds, and 0 for tracking, information requests or no action.\n"
            "3. Draft customer_message (maximum 120 words, plain text) assuming the action is approved; use the customer's name "
            "and the order id, and be specific about what happens next and when."
        ),
        expected_output=(
            "A single JSON object (no markdown fences; every string value on one line, no line breaks) with exactly these fields:\n"
            f"recommended_action (one of: {ACTIONS}), refund_amount_inr (number), rationale (two sentences), "
            "customer_message (string)."
        ),
        agent=resolution_agent, context=[identify, interpret],
    )

    escalate = Task(
        description=(
            "Decide whether the recommended resolution can be applied automatically.\n"
            "1. Call evaluate_approval exactly once, passing the full JSON from task 1 as identification_json, task 2 as "
            "policy_json, task 3 as resolution_json, and the order id {order_id}.\n"
            "2. Adopt the tool's decision, approver and risk_score exactly.\n"
            "3. Rewrite the tool's reasons as 1-3 plain-language bullets.\n"
            "4. Produce final_customer_message: if AUTO_APPROVE, use the drafted message from task 3 as is; if MANUAL_APPROVAL, "
            "rewrite it so the customer is told the request has been passed to a specialist for approval, with a response within "
            "24 hours, and nothing is promised yet; if ESCALATE_FRAUD_REVIEW, send a neutral message saying the request is under "
            "review and the team will be in touch within 48 hours, without mentioning fraud.\n"
            "5. Write an approver_note (max 2 sentences) for the human approver when the decision is not AUTO_APPROVE, otherwise null."
        ),
        expected_output=(
            "A single JSON object (no markdown fences; every string value on one line, no line breaks) with exactly these fields:\n"
            "decision (AUTO_APPROVE | MANUAL_APPROVAL | ESCALATE_FRAUD_REVIEW), approver (string or null), risk_score (number), "
            "reasons (list of strings), final_customer_message (string), approver_note (string or null)."
        ),
        agent=escalation_agent, context=[identify, interpret, recommend],
    )
    return [identify, interpret, recommend, escalate]


def build_crew(main_llm, reason_llm, verbose=VERBOSE):
    agents = build_agents(main_llm, reason_llm, verbose)
    return Crew(agents=list(agents), tasks=build_tasks(agents), process=Process.sequential,
                verbose=verbose, memory=False, cache=False)

_crew_check = build_crew(primary_llm, reasoning_llm)
print("Crew built:", [a.role for a in _crew_check.agents])

Crew built: ['Order Issue Identification Agent', 'Policy Interpretation Agent', 'Resolution Recommendation Agent', 'Escalation Agent']


## Section C — Workflow Execution

Seven representative queries. Four are routine and should be approved automatically; three are edge cases — a policy exception, a high-value refund and a repeat refunder with a disputed delivery — that must go to a human. Each query carries an `expected_decision` so false positives and false negatives can be measured.

C.1 tests the approval policy on hand-labelled inputs without any model calls, so policy errors can be separated from identification errors. C.2 defines the run harness, C.3 traces one edge case with full agent output, and C.4 runs every query.

In [8]:
# Sample customer queries with expected approval decisions

SAMPLE_QUERIES = [
    {"ticket_id": "Q-2001", "order_id": "ORD-1001",
     "message": "Hi, my dinner set was supposed to arrive on 5 Nov and it still hasn't come. Where is it?",
     "expected_decision": "AUTO_APPROVE"},
    {"ticket_id": "Q-2002", "order_id": "ORD-1002",
     "message": "The T-shirt I ordered is too small. I'd like to return it and get my money back.",
     "expected_decision": "AUTO_APPROVE"},
    {"ticket_id": "Q-2003", "order_id": "ORD-1003",
     "message": "The blender arrived with a cracked jar. I've attached photos. Can you send a replacement?",
     "expected_decision": "AUTO_APPROVE"},
    {"ticket_id": "Q-2004", "order_id": "ORD-1004",
     "message": "I bought these headphones a few weeks ago and the sound is tinny. I want a refund.",
     "expected_decision": "MANUAL_APPROVAL"},
    {"ticket_id": "Q-2005", "order_id": "ORD-1005",
     "message": "The laptop box was crushed and the screen has a crack. Photos attached. Please refund me.",
     "expected_decision": "MANUAL_APPROVAL"},
    {"ticket_id": "Q-2006", "order_id": "ORD-1006",
     "message": "I never received my smartwatch even though the app says delivered. Refund me now.",
     "expected_decision": "ESCALATE_FRAUD_REVIEW"},
    {"ticket_id": "Q-2007", "order_id": "ORD-1007",
     "message": "Something is wrong with my order. Please fix it.",
     "expected_decision": "MANUAL_APPROVAL"},
]
print(f"{len(SAMPLE_QUERIES)} sample queries loaded.")

7 sample queries loaded.


In [9]:
# C.1 Policy unit test: rule engine on hand-labelled inputs (no model calls)
# Labels represent a correct manual handling of each query.

HAND_LABELS = {
    "Q-2001": ({"issue_category": "delayed_delivery", "confidence": 0.9, "evidence_provided": False},
               {"within_return_window": None, "policy_exception_requested": False},
               {"recommended_action": "track_and_wait", "refund_amount_inr": 0}),
    "Q-2002": ({"issue_category": "return_request", "confidence": 0.95, "evidence_provided": False},
               {"within_return_window": True, "policy_exception_requested": False},
               {"recommended_action": "refund_and_return", "refund_amount_inr": 1299}),
    "Q-2003": ({"issue_category": "damaged_product", "confidence": 0.95, "evidence_provided": True},
               {"within_return_window": True, "policy_exception_requested": False},
               {"recommended_action": "replacement", "refund_amount_inr": 3499}),
    "Q-2004": ({"issue_category": "refund_request", "confidence": 0.85, "evidence_provided": False},
               {"within_return_window": False, "policy_exception_requested": True},
               {"recommended_action": "full_refund", "refund_amount_inr": 2499}),
    "Q-2005": ({"issue_category": "damaged_product", "confidence": 0.95, "evidence_provided": True},
               {"within_return_window": True, "policy_exception_requested": False},
               {"recommended_action": "full_refund", "refund_amount_inr": 68999}),
    "Q-2006": ({"issue_category": "not_received", "confidence": 0.85, "evidence_provided": False},
               {"within_return_window": True, "policy_exception_requested": False},
               {"recommended_action": "full_refund", "refund_amount_inr": 4999}),
    "Q-2007": ({"issue_category": "other", "confidence": 0.3, "evidence_provided": False},
               {"within_return_window": True, "policy_exception_requested": False},
               {"recommended_action": "request_more_information", "refund_amount_inr": 0}),
}

print(f"{'Ticket':<8}{'Expected':<24}{'Rule engine':<24}{'Score':<7}Match")
print("-" * 70)
mismatches = 0
for q in SAMPLE_QUERIES:
    ident, pol, res = HAND_LABELS[q["ticket_id"]]
    out = evaluate_approval_rules(ident, pol, res, ORDERS[q["order_id"]])
    match = out["decision"] == q["expected_decision"]
    mismatches += (not match)
    print(f"{q['ticket_id']:<8}{q['expected_decision']:<24}{out['decision']:<24}{out['risk_score']:<7}{'yes' if match else 'NO'}")
print(f"\nPolicy unit test: {len(SAMPLE_QUERIES) - mismatches}/{len(SAMPLE_QUERIES)} decisions match expectations.")

Ticket  Expected                Rule engine             Score  Match
----------------------------------------------------------------------
Q-2001  AUTO_APPROVE            AUTO_APPROVE            0      yes
Q-2002  AUTO_APPROVE            AUTO_APPROVE            0      yes
Q-2003  AUTO_APPROVE            AUTO_APPROVE            0      yes
Q-2004  MANUAL_APPROVAL         MANUAL_APPROVAL         30     yes
Q-2005  MANUAL_APPROVAL         MANUAL_APPROVAL         60     yes
Q-2006  ESCALATE_FRAUD_REVIEW   ESCALATE_FRAUD_REVIEW   100    yes
Q-2007  MANUAL_APPROVAL         MANUAL_APPROVAL         30     yes

Policy unit test: 7/7 decisions match expectations.


In [10]:
# C.2 Run harness: kick off the crew, fall back on provider failure, parse outputs, audit the decision

import time
from concurrent.futures import ThreadPoolExecutor

TASK_NAMES = ["identification", "policy", "resolution", "escalation"]

def kickoff_in_thread(crew, inputs: dict):
    """Run the crew in a worker thread; Jupyter keeps an event loop running on the main thread."""
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(crew.kickoff, inputs=inputs).result()

def parse_output(raw, name):
    try:
        return extract_json(raw)
    except Exception as error:
        return {"_parse_error": f"{name}: {error}", "_raw": str(raw)}

def run_query(query: dict, verbose: bool = VERBOSE) -> dict:
    """Run one customer query through the crew and return an auditable record."""
    inputs = {k: query[k] for k in ("ticket_id", "order_id", "message")}
    model_sets = [("primary", primary_llm, reasoning_llm)]
    if fallback_llm is not None:
        model_sets.append(("fallback", fallback_llm, fallback_llm))
    attempts = [(f"{label} try {n}", main, reason) for label, main, reason in model_sets for n in (1, 2)]

    result, used, errors = None, None, []
    started = datetime.now(timezone.utc)
    for label, main, reason in attempts:
        try:
            result = kickoff_in_thread(build_crew(main, reason, verbose), inputs)
            used = label
            break
        except Exception as error:
            errors.append(f"{label}: {str(error)[:300]}")
            print(f"  [{query['ticket_id']}] {label} failed -> {str(error)[:120]}")
            time.sleep(5)
    if result is None:
        raise RuntimeError(f"All model attempts failed for {query['ticket_id']}: {errors}")

    outputs = {name: parse_output(t.raw, name) for name, t in zip(TASK_NAMES, result.tasks_output)}
    identification, policy, resolution, escalation = (outputs[n] for n in TASK_NAMES)

    # Audit: the rule engine is the source of truth for the decision, provided every upstream
    # output parsed. If one did not, the Escalation agent's own decision stands and the record says so.
    parsed_ok = all("_parse_error" not in o for o in (identification, policy, resolution))
    rule = evaluate_approval_rules(identification, policy, resolution, ORDERS.get(query["order_id"], {}))
    if not parsed_ok:
        rule = {"decision": escalation.get("decision", "UNKNOWN"), "approver": escalation.get("approver"),
                "risk_score": escalation.get("risk_score"), "override": None,
                "reasons": ["audit skipped: an upstream output could not be parsed; Escalation agent decision kept"]}
    return {
        "ticket_id": query["ticket_id"], "order_id": query["order_id"], "message": query["message"],
        "expected_decision": query["expected_decision"], "models_used": used,
        "elapsed_s": round((datetime.now(timezone.utc) - started).total_seconds(), 1),
        "identification": identification, "policy": policy, "resolution": resolution,
        "escalation_agent": escalation, "rule_engine": rule,
        "final_decision": rule["decision"], "approver": rule["approver"],
        "agent_matches_rule_engine": escalation.get("decision") == rule["decision"],
        "all_outputs_parsed": parsed_ok,
    }

def print_report(r: dict):
    i, p, s, e = r["identification"], r["policy"], r["resolution"], r["escalation_agent"]
    print("=" * 96)
    print(f"{r['ticket_id']}  |  {r['order_id']}  |  models: {r['models_used']}  |  {r['elapsed_s']}s")
    print(f"Customer: {r['message']}")
    print("-" * 96)
    print(f"[1 Identification]  category={i.get('issue_category')}  evidence={i.get('evidence_provided')}  confidence={i.get('confidence')}")
    print(f"                    request: {i.get('customer_request')}")
    print(f"[2 Policy]          window={p.get('return_window_days')}d  since_delivery={p.get('days_since_delivery')}  "
          f"within_window={p.get('within_return_window')}  exception={p.get('policy_exception_requested')}  rules={p.get('applicable_rules')}")
    print(f"                    {p.get('policy_summary')}")
    print(f"[3 Resolution]      action={s.get('recommended_action')}  amount=Rs {s.get('refund_amount_inr')}")
    print(f"                    {s.get('rationale')}")
    print(f"[4 Escalation]      agent={e.get('decision')} (score {e.get('risk_score')})  ->  rule engine={r['final_decision']} "
          f"(score {r['rule_engine']['risk_score']}, approver {r['approver']})  "
          f"{'consistent' if r['agent_matches_rule_engine'] else 'MISMATCH: rule engine wins'}")
    for reason in r["rule_engine"]["reasons"]:
        print(f"                    - {reason}")
    if e.get("approver_note"):
        print(f"                    approver note: {e.get('approver_note')}")
    print(f"[Audit]             expected={r['expected_decision']}  {'OK' if r['final_decision'] == r['expected_decision'] else 'DIFFERS'}")
    print()
    print(str(e.get("final_customer_message", e.get("_raw", ""))).strip())
    print()
    for key in ("identification", "policy", "resolution", "escalation_agent"):
        if "_parse_error" in r[key]:
            print(f"                    WARNING {r[key]['_parse_error']}")
    print("=" * 96, "\n")

In [11]:
# C.3 Full trace of one edge case (high-value damaged laptop) showing tool calls and handoffs

demo = run_query(SAMPLE_QUERIES[4], verbose=True)
print_report(demo)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ae9cf6cb-403c-4b7a-bd5a-b6e6729a4368                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Identify the issue in this customer query.                                                               │
│                                                                                                                 │
│  Ticket id: Q-2005                                                                                              │
│  Order id: ORD-1005                                                                                             │
│  Customer message:                                                                                              │
│  The laptop box was crushed and the screen has a crack. Photos attached. Please refund me.                      │
│                                                                                                                 │
│  1. Call lookup_order with the order id.                                                                        │
│  2. Classify the issue using only what the customer wrote and what the order record shows.                      │
│  3. Set evidence_provided = true only if the customer says they attached or can share photos or video.          │
│  4. Set confidence between 0 and 1; a query with no clear problem statement must score below 0.6.               │
│  ID: a1781911-9092-4b9a-bcd7-6862d0e96d94                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Order Issue Identification Agent                                                                        │
│                                                                                                                 │
│  Task: Identify the issue in this customer query.                                                               │
│                                                                                                                 │
│  Ticket id: Q-2005                                                                                              │
│  Order id: ORD-1005                                                                                             │
│  Customer message:                                                                                              │
│  The laptop box was crushed and the screen has a crack. Photos attached. Please refund me.                      │
│                                                                                                                 │
│  1. Call lookup_order with the order id.                                                                        │
│  2. Classify the issue using only what the customer wrote and what the order record shows.                      │
│  3. Set evidence_provided = true only if the customer says they attached or can share photos or video.          │
│  4. Set confidence between 0 and 1; a query with no clear problem statement must score below 0.6.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: lookup_order                                                                                             │
│  Args: {'order_id': 'ORD-1005'}                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: lookup_order                                                                                             │
│  Output: {"order_id": "ORD-1005", "customer": "Priya Iyer", "item": "Laptop 14-inch, 16 GB RAM",                │
│  "item_category": "electronics", "amount_inr": 68999, "order_date": "2026-11-04", "estimated_delivery":         │
│  "2026-11-07", "delivered_date": "2026-11-07", "status": "delivered", "payment_method": "card",                 │
│  "refunds_last_90d": 0, "days_late": 0, "today": "2026-11-09"}                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


... (20 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Decide whether the recommended resolution can be applied automatically.                                  │
│  1. Call evaluate_approval exactly once, passing the full JSON from task 1 as identification_json, task 2 as    │
│  policy_json, task 3 as resolution_json, and the order id ORD-1005.                                             │
│  2. Adopt the tool's decision, approver and risk_score exactly.                                                 │
│  3. Rewrite the tool's reasons as 1-3 plain-language bullets.                                                   │
│  4. Produce final_customer_message: if AUTO_APPROVE, use the drafted message from task 3 as is; if              │
│  MANUAL_APPROVAL, rewrite it so the customer is told the request has been passed to a specialist for approval,  │
│  with a response within 24 hours, and nothing is promised yet; if ESCALATE_FRAUD_REVIEW, send a neutral         │
│  message saying the request is under review and the team will be in touch within 48 hours, without mentioning   │
│  fraud.                                                                                                         │
│  5. Write an approver_note (max 2 sentences) for the human approver when the decision is not AUTO_APPROVE,      │
│  otherwise null.                                                                                                │
│  Agent: Escalation Agent                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ae9cf6cb-403c-4b7a-bd5a-b6e6729a4368                                                                       │
│  Final Output: {"decision": "MANUAL_APPROVAL", "approver": "team lead", "risk_score": 60, "reasons": ["The      │
│  refund amount of Rs 68,999 is above the Rs 15,000 threshold, so policy rule 6 requires team lead approval.",   │
│  "The request itself is valid: the order is only 2 days past delivery, well inside the 15-day electronics       │
│  return window, and photo/video evidence confirms the damage.", "Because the amount exceeds the approval        │
│  limit, the refund cannot be applied automatically and must be signed off by a team lead before any money is    │
│  returned."], "final_customer_message": "Dear Customer, thank you for contacting us about order ORD-1005 and    │
│  for sending the photos of the crushed box and cracked screen on your Laptop 14-inch. We're sorry this          │
│  happened. Your request for a full refund of Rs 68,999 has been passed to a specialist for approval, and you    │
│  can expect a response within 24 hours. Please note that nothing has been confirmed yet and no refund has been  │
│  issued at this stage. We'll be in touch as soon as the review is complete. Thank you for your patience.",      │
│  "approver_note": "Refund of Rs 68,999 for damaged laptop ORD-1005 is within the 15-day return window with      │
│  photo evidence, but exceeds the Rs 15,000 limit and needs team lead sign-off. Please confirm the full refund   │
│  to the original card (5-7 working days) or offer the free replacement instead."}                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Q-2005  |  ORD-1005  |  models: primary try 1  |  9.7s
Customer: The laptop box was crushed and the screen has a crack. Photos attached. Please refund me.
------------------------------------------------------------------------------------------------
[1 Identification]  category=damaged_product  evidence=True  confidence=0.92
                    request: The customer wants a refund because the laptop box arrived crushed and the screen has a crack.
[2 Policy]          window=15d  since_delivery=2  within_window=True  exception=False  rules=['1', '2', '3', '6']
                    The electronics return window is 15 days from delivery, and with only 2 days elapsed the request is well within policy. Because the laptop arrived damaged and photo/video evidence was provided, the customer may choose either a free replacement or a full refund to the original card, subject to the required senior agent and team lead approvals for the Rs 68,999 amount.
[3 Resolution]      action=full_refund  amo

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [12]:
# C.4 Run all sample queries

RESULTS = []
for query in SAMPLE_QUERIES:
    print(f"Running {query['ticket_id']} ...")
    try:
        record = run_query(query, verbose=False)
    except Exception as error:
        record = {"ticket_id": query["ticket_id"], "order_id": query["order_id"], "message": query["message"],
                  "expected_decision": query["expected_decision"], "final_decision": "RUN_ERROR", "approver": None,
                  "agent_matches_rule_engine": False, "error": str(error)[:400], "identification": {}, "policy": {},
                  "resolution": {}, "escalation_agent": {}, "rule_engine": {"reasons": [], "risk_score": "-"},
                  "models_used": "-", "elapsed_s": 0}
        print(f"  RUN_ERROR: {record['error']}")
    RESULTS.append(record)

for record in RESULTS:
    if record["final_decision"] != "RUN_ERROR":
        print_report(record)

Running Q-2001 ...
Running Q-2002 ...
Running Q-2003 ...
Running Q-2004 ...
Running Q-2005 ...
Running Q-2006 ...


ERROR:crewai.flow.runtime:Error executing listener call_llm_and_parse: The provider returned no content for model 'deepseek-ai/DeepSeek-V4.1-Flash'.


Running Q-2007 ...
Q-2001  |  ORD-1001  |  models: primary try 1  |  9.9s
Customer: Hi, my dinner set was supposed to arrive on 5 Nov and it still hasn't come. Where is it?
------------------------------------------------------------------------------------------------
[1 Identification]  category=delayed_delivery  evidence=False  confidence=0.9
                    request: The customer wants to know where her ceramic dinner set is, since it was supposed to arrive on 5 Nov and has not been delivered.
[2 Policy]          window=30d  since_delivery=None  within_window=None  exception=False  rules=['1', '4']
                    The ceramic dinner set is a home item with a 30-day return window that starts on the delivery date, but the order has not been delivered yet, so no return window is running and no return or refund can be processed while it is in transit. Because the order is 4 days past its estimated date it counts as delayed under Rule 4, so the customer's options are to wait for 

## Section D — Results and Evaluation

The audit compares, for every query, the expected decision, the rule engine's decision on the agents' structured outputs, and the Escalation agent's own stated decision. A false negative (a case that needed approval but was auto-approved) is the dangerous error; a false positive (a routine case sent for approval) is the costly one.

In [13]:
# D.1 Decision audit: approval accuracy, false positives / negatives, agent-vs-rule-engine agreement

import pandas as pd

NEEDS_HUMAN = {"MANUAL_APPROVAL", "ESCALATE_FRAUD_REVIEW"}

rows = []
for r in RESULTS:
    exp, got = r["expected_decision"], r["final_decision"]
    if got == "RUN_ERROR":
        verdict = "run error"
    elif got == exp:
        verdict = "correct"
    elif got in NEEDS_HUMAN and exp not in NEEDS_HUMAN:
        verdict = "false positive (over-escalated)"
    elif got not in NEEDS_HUMAN and exp in NEEDS_HUMAN:
        verdict = "FALSE NEGATIVE (under-escalated)"
    else:
        verdict = "wrong queue"
    rows.append({
        "ticket": r["ticket_id"], "order": r["order_id"],
        "category": r["identification"].get("issue_category"),
        "confidence": r["identification"].get("confidence"),
        "action": r["resolution"].get("recommended_action"),
        "amount_inr": r["resolution"].get("refund_amount_inr"),
        "score": r["rule_engine"].get("risk_score"),
        "expected": exp, "final": got, "approver": r["approver"],
        "agent==rules": r["agent_matches_rule_engine"], "verdict": verdict,
        "models": r["models_used"], "secs": r["elapsed_s"],
    })

summary = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 40); pd.set_option("display.width", 200)
display(summary)

print(f"\nDecision accuracy: {(summary['verdict'] == 'correct').sum()}/{len(summary)}")
print(f"False negatives (dangerous): {summary['verdict'].str.startswith('FALSE').sum()}")
print(f"False positives (costly):    {summary['verdict'].str.startswith('false').sum()}")
print(f"Escalation agent agreed with rule engine on {summary['agent==rules'].sum()}/{len(summary)} queries")

with open("week12_ecommerce_results.json", "w") as fh:
    json.dump(RESULTS, fh, indent=2, default=str)
print("Full records saved to week12_ecommerce_results.json")

,ticket,order,category,confidence,action,amount_inr,score,expected,final,approver,agent==rules,verdict,models,secs
0,Q-2001,ORD-1001,delayed_delivery,0.90,track_and_wait,0,0,AUTO_APPROVE,AUTO_APPROVE,NaN,True,correct,primary try 1,9.9
1,Q-2002,ORD-1002,return_request,0.85,refund_and_return,1299,0,AUTO_APPROVE,AUTO_APPROVE,NaN,True,correct,primary try 1,11.9
2,Q-2003,ORD-1003,damaged_product,0.95,replacement,3499,0,AUTO_APPROVE,AUTO_APPROVE,NaN,True,correct,primary try 1,9.9
3,Q-2004,ORD-1004,refund_request,0.80,full_refund,2499,30,MANUAL_APPROVAL,MANUAL_APPROVAL,senior agent,True,correct,primary try 1,20.3
4,Q-2005,ORD-1005,damaged_product,0.92,full_refund,68999,60,MANUAL_APPROVAL,MANUAL_APPROVAL,team lead,True,correct,primary try 1,12.1
5,Q-2006,ORD-1006,not_received,0.85,request_more_information,0,100,ESCALATE_FRAUD_REVIEW,ESCALATE_FRAUD_REVIEW,fraud-review team,True,correct,primary try 1,38.7
6,Q-2007,ORD-1007,other,0.20,request_more_information,0,30,MANUAL_APPROVAL,MANUAL_APPROVAL,senior agent,True,correct,primary try 1,11.4



Decision accuracy: 7/7
False negatives (dangerous): 0
False positives (costly):    0
Escalation agent agreed with rule engine on 7/7 queries
Full records saved to week12_ecommerce_results.json


### D.2 Summary
The system handles routine post-purchase queries end to end — tracking a delayed order, accepting an in-window return, replacing a damaged item with evidence — and refers the cases that carry risk to a human: refunds above the approval limits, requests outside the return window, disputed deliveries, vague queries and customers with a pattern of refunds. Because the approval policy is code, each decision is reproducible and carries its reasons; the language models do the reading, policy reasoning and writing. The architecture summary submitted with this notebook describes the agent roles, task flow, escalation logic and design rationale in full.